# Paper-faithful SVD style personalization on VAR-CLIP

This notebook ports the inference-time method from **A Training-Free Style-Personalization via SVD-Based Feature Decomposition** (CVPR 2026, arXiv:2507.04482v2) from Infinity-2B to the released VAR-CLIP checkpoint.

The port keeps the paper's key design:

1. Encode one reference image into cumulative multi-scale VAE features.
2. Run an unmodified **content path** and an edited **generation path** jointly under the exact same text prompt.
3. Apply Principal Feature Blending (PFB) to the third accumulated feature $F_3$.
4. From the next causally available prediction onward, replace generation-path self-attention queries and keys with content-path queries and keys while retaining generation-path values (SAC).
5. Freeze every model parameter. There is no training, optimization, LoRA, or VAE token replacement after generation.

VAR-CLIP has 10 scales `(1,2,3,4,5,6,8,10,13,16)` and generates 256x256 images, whereas the paper's Infinity-2B has 12 steps and generates 1024x1024 images. Therefore this is a faithful **method port**, not a claim that VAR-CLIP will reproduce Infinity-2B image quality.

## 1. Colab setup

Select a GPU runtime before running. The notebook clones the official VAR-CLIP repository and downloads its released checkpoint, the official VAR VAE, and OpenAI CLIP ViT-L/14.

In [ ]:
!nvidia-smi

import os
import subprocess
from pathlib import Path

assert (
    os.path.exists('/usr/local/cuda')
    or os.environ.get('COLAB_GPU')
    or Path('/kaggle/working').exists()
), 'No hosted GPU runtime detected. Select a GPU runtime and reconnect.'

if Path('/kaggle/working').exists():
    RUNTIME_ROOT = Path('/kaggle/working')
elif Path('/content').exists():
    RUNTIME_ROOT = Path('/content')
else:
    RUNTIME_ROOT = Path.cwd()

VAR_CLIP_REPO = 'https://github.com/daixiangzi/VAR-CLIP.git'
VAR_CLIP_DIR = RUNTIME_ROOT / 'VAR-CLIP'
OUTPUT_DIR = RUNTIME_ROOT / 'VAR_CLIP_outputs' / 'paper_faithful_svd_pfb_sac'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not VAR_CLIP_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', VAR_CLIP_REPO, str(VAR_CLIP_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(VAR_CLIP_DIR), 'fetch', 'origin', 'master'], check=True)
    subprocess.run(['git', '-C', str(VAR_CLIP_DIR), 'reset', '--hard', 'origin/master'], check=True)

os.chdir(VAR_CLIP_DIR)
print('VAR-CLIP source:', VAR_CLIP_DIR)
print('outputs:', OUTPUT_DIR)

In [ ]:
!pip -q install gdown huggingface_hub einops typed-argument-parser pytz open_clip_torch pandas tqdm

import gc
import importlib
import math
import random
import sys
import types

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from tqdm.auto import tqdm

assert torch.cuda.is_available(), 'Select a GPU runtime, reconnect, and rerun.'
device = 'cuda'
print('torch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from huggingface_hub import hf_hub_download
import gdown

PRETRAINED_DIR = VAR_CLIP_DIR / 'pretrained'
LOCAL_OUTPUT_DIR = VAR_CLIP_DIR / 'local_output'
PRETRAINED_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vae_path = Path(hf_hub_download(
    repo_id='FoundationVision/var',
    filename='vae_ch160v4096z32.pth',
    local_dir=PRETRAINED_DIR,
))

clip_path = PRETRAINED_DIR / 'ViT-L-14.pt'
clip_url = (
    'https://openaipublic.azureedge.net/clip/models/'
    'b8cca3fd41ae0c99ba7e8951adf17d267cdb84cd88be6f7c2e0eca1737a03836/ViT-L-14.pt'
)
if not clip_path.exists() or clip_path.stat().st_size < 500_000_000:
    subprocess.run(['wget', '-c', '--show-progress', '-O', str(clip_path), clip_url], check=True)

var_clip_path = LOCAL_OUTPUT_DIR / 'ar-ckpt-last.pth'
var_clip_url = 'https://drive.google.com/file/d/10gSxvaKaNKJcnqFhU7hQywU28w3nbgoV/view?usp=sharing'
if not var_clip_path.exists() or var_clip_path.stat().st_size < 100_000_000:
    gdown.download(url=var_clip_url, output=str(var_clip_path), fuzzy=True)

assert vae_path.exists(), vae_path
assert clip_path.exists() and clip_path.stat().st_size > 500_000_000, 'Incomplete CLIP checkpoint.'
assert var_clip_path.exists() and var_clip_path.stat().st_size > 100_000_000, 'Incomplete VAR-CLIP checkpoint.'
print('VAE:', vae_path)
print('CLIP:', clip_path)
print('VAR-CLIP:', var_clip_path)

In [ ]:
# PyTorch 2.6+ needs weights_only=False for the trusted OpenAI TorchScript archive.
clip_source = VAR_CLIP_DIR / 'models' / 'clip.py'
clip_text = clip_source.read_text()
old_call = "pretrained='pretrained/ViT-L-14.pt')"
new_call = "pretrained='pretrained/ViT-L-14.pt', weights_only=False)"
if old_call in clip_text:
    clip_source.write_text(clip_text.replace(old_call, new_call, 1))

setattr(torch.nn.Linear, 'reset_parameters', lambda self: None)
setattr(torch.nn.LayerNorm, 'reset_parameters', lambda self: None)

from clip_util import CLIPWrapper
from models.clip import clip_vit_l14
from tokenizer import tokenize
from models import build_vae_var
from models.basic_var import slow_attn
from models.helpers import sample_with_top_k_top_p_

MODEL_DEPTH = 16
PATCH_NUMS = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)

vae, var_clip = build_vae_var(
    V=4096,
    Cvae=32,
    ch=160,
    share_quant_resi=4,
    device=device,
    patch_nums=PATCH_NUMS,
    n_cond_embed=768,
    depth=MODEL_DEPTH,
    shared_aln=False,
)

clip_model = CLIPWrapper(clip_vit_l14(pretrained=True).to(device).eval(), normalize=True)
vae.load_state_dict(torch.load(vae_path, map_location='cpu', weights_only=False), strict=True)
checkpoint = torch.load(var_clip_path, map_location='cpu', weights_only=False)
var_clip.load_state_dict(checkpoint['trainer']['var_wo_ddp'], strict=True)
del checkpoint

vae.eval()
var_clip.eval()
for model in (vae, var_clip, clip_model.clip):
    for parameter in model.parameters():
        parameter.requires_grad_(False)

torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')
gc.collect()
torch.cuda.empty_cache()
print('Frozen VAR-CLIP-d16, VAR VAE, and CLIP ViT-L/14 are ready.')

## 2. Reference styles and cross-content prompts

Each prompt follows the paper's `<content> in <style>` rule. The target object intentionally differs from the object or scene in the reference image, so successful output cannot be explained by copying reference content.

In [ ]:
STYLE_WORKSPACE_REPO = 'https://github.com/LeeHoang2710/Style-Transfer-Experiment.git'
STYLE_WORKSPACE = RUNTIME_ROOT / 'VAR_Style_Transfer_Workspace'

if not STYLE_WORKSPACE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', STYLE_WORKSPACE_REPO, str(STYLE_WORKSPACE)], check=True)
else:
    subprocess.run(['git', '-C', str(STYLE_WORKSPACE), 'fetch', 'origin', 'main'], check=True)
    subprocess.run(['git', '-C', str(STYLE_WORKSPACE), 'reset', '--hard', 'origin/main'], check=True)

STYLE_DIR = STYLE_WORKSPACE / 'style'
DEMO_CASES = [
    {
        'name': 'elephant in dog-sketch style',
        'prompt': 'an elephant, animal, in pencil sketch style',
        'style_path': STYLE_DIR / 'Sketch' / 'S005.png',
        'style_label': 'pencil sketch reference (dog)',
    },
    {
        'name': 'teapot in car-line-art style',
        'prompt': 'a teapot, object, in short line drawing style',
        'style_path': STYLE_DIR / 'LineDrawing' / 'LD001.png',
        'style_label': 'line drawing reference (car)',
    },
    {
        'name': 'lighthouse in watercolor style',
        'prompt': 'a lighthouse beside the sea, building, in watercolor painting style',
        'style_path': STYLE_DIR / 'WaterColor' / 'WC001.png',
        'style_label': 'watercolor reference (landscape)',
    },
    {
        'name': 'steam train in oil-painting style',
        'prompt': 'a steam train, vehicle, in oil painting style',
        'style_path': STYLE_DIR / 'OilPainting' / 'OP005.png',
        'style_label': 'oil painting reference (mountain)',
    },
    {
        'name': 'red fox in mosaic style',
        'prompt': 'a red fox, animal, in mosaic tile artwork style',
        'style_path': STYLE_DIR / 'PixelArt' / 'PA012.png',
        'style_label': 'mosaic reference (dog)',
    },
]

for case in DEMO_CASES:
    assert case['style_path'].exists(), case['style_path']

print(pd.DataFrame([{k: str(v) for k, v in case.items()} for case in DEMO_CASES]).to_string(index=False))

## 3. Exact Principal Feature Blending

For a cumulative feature map $F \in \mathbb{R}^{C\times H\times W}$, flatten spatial dimensions to $F \in \mathbb{R}^{C\times HW}$ and compute

$$F=U\Sigma V^\top,$$

$$\Phi(F)=UW\Sigma V^\top, \qquad W_{ii}=\exp(-i\alpha).$$

At the third accumulated feature:

$$F_3^{gen}\leftarrow\Phi(F_3^{sty})+\left(F_3^{gen}-\Phi(F_3^{gen})\right).$$

No centering, covariance transform, normalization, or learned projection is added; those would change the paper's method.

In [ ]:
def load_reference_image(path, size=256):
    image = Image.open(path).convert('RGB')
    image = ImageOps.fit(image, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    image_01 = torchvision.transforms.functional.to_tensor(image).unsqueeze(0).to(device)
    return image_01.mul(2).sub(1), image_01, image


@torch.no_grad()
def extract_multiscale_style_features(image_m11):
    latent = vae.quant_conv(vae.encoder(image_m11))
    features = vae.quantize.f_to_idxBl_or_fhat(latent, to_fhat=True)
    assert len(features) == len(PATCH_NUMS)
    assert all(feature.shape[-2:] == (PATCH_NUMS[-1], PATCH_NUMS[-1]) for feature in features)
    return features


def phi_svd(feature_bchw, alpha=1.0, rank=None):
    '''Paper Eq. (5): exponentially reweighted SVD, applied per sample.'''
    original_dtype = feature_bchw.dtype
    batch, channels, height, width = feature_bchw.shape
    outputs = []

    for batch_id in range(batch):
        matrix = feature_bchw[batch_id].detach().float().reshape(channels, height * width)
        u, singular_values, vh = torch.linalg.svd(matrix, full_matrices=False)
        available_rank = singular_values.numel()
        used_rank = available_rank if rank is None else min(int(rank), available_rank)
        weights = torch.exp(
            -float(alpha) * torch.arange(used_rank, device=matrix.device, dtype=matrix.dtype)
        )
        weighted_s = singular_values[:used_rank] * weights
        reconstructed = (u[:, :used_rank] * weighted_s.unsqueeze(0)) @ vh[:used_rank]
        outputs.append(reconstructed.reshape(channels, height, width))

    return torch.stack(outputs).to(dtype=original_dtype)


def principal_feature_blend(generation_feature, style_feature, alpha=1.0, rank=None):
    '''Paper Eq. (6): replace principal style component, preserve generation residual.'''
    if generation_feature.shape != style_feature.shape:
        raise ValueError(f'PFB shape mismatch: {generation_feature.shape} vs {style_feature.shape}')
    style_feature = style_feature.to(generation_feature)
    return phi_svd(style_feature, alpha=alpha, rank=rank) + (
        generation_feature - phi_svd(generation_feature, alpha=alpha, rank=rank)
    )


def apply_feature_edit(generation_feature, style_feature, mode, alpha=1.0, rank=None):
    if mode == 'none':
        return generation_feature
    if mode == 'replace':
        return style_feature.to(generation_feature)
    if mode == 'pfb':
        return principal_feature_blend(generation_feature, style_feature, alpha=alpha, rank=rank)
    raise ValueError(f'Unknown edit mode: {mode}')

## 4. Exact Structural Attention Correction

VAR-CLIP has self-attention but no Infinity-style text cross-attention. Text enters through CLIP conditioning and AdaLN. SAC therefore maps cleanly to each VAR self-attention block:

$$Q_s^{gen}\leftarrow Q_s^{con},\qquad K_s^{gen}\leftarrow K_s^{con},$$

while $V_s^{gen}$ remains untouched.

The joint inference batch is ordered as:

`[content_cond, generation_cond, content_uncond, generation_uncond]`.

PFB edits $F_3$ only after residual $R_3$ has been sampled. Because next-scale generation is causal, the first attention computation that can consume edited $F_3$ predicts $R_4$. Thus paper stage `s=3` corresponds to zero-based PFB feature index `2`, while executable SAC begins at zero-based prediction index `3`.

In [ ]:
class SACController:
    def __init__(self, base_batch=1):
        self.base_batch = base_batch
        self.active = False
        self.total_calls = 0
        self.max_q_copy_error = 0.0
        self.max_k_copy_error = 0.0

    def reset_statistics(self):
        self.total_calls = 0
        self.max_q_copy_error = 0.0
        self.max_k_copy_error = 0.0


def _sac_attention_forward(attention, x, attn_bias):
    batch4, length, channels = x.shape
    qkv = F.linear(
        x,
        attention.mat_qkv.weight,
        torch.cat((attention.q_bias, attention.zero_k_bias, attention.v_bias)),
    ).view(batch4, length, 3, attention.num_heads, attention.head_dim)

    # Force B,H,L,d layout so Q/K are replaced before K/V cache concatenation.
    q, k, v = qkv.permute(2, 0, 3, 1, 4).unbind(dim=0)
    if attention.attn_l2_norm:
        scale_multiplier = attention.scale_mul_1H11.clamp_max(attention.max_scale_mul).exp()
        q = F.normalize(q, dim=-1).mul(scale_multiplier)
        k = F.normalize(k, dim=-1)

    controller = getattr(attention, '_paper_sac_controller', None)
    if controller is not None and controller.active:
        b = controller.base_batch
        if batch4 != 4 * b:
            raise RuntimeError(f'SAC expected joint batch {4 * b}, received {batch4}.')

        # Order: content conditional, generation conditional,
        #        content unconditional, generation unconditional.
        q = torch.cat((q[:b], q[:b], q[2*b:3*b], q[2*b:3*b]), dim=0)
        k = torch.cat((k[:b], k[:b], k[2*b:3*b], k[2*b:3*b]), dim=0)

        controller.total_calls += 1
        controller.max_q_copy_error = max(
            controller.max_q_copy_error,
            float((q[b:2*b] - q[:b]).abs().max().detach().cpu()),
            float((q[3*b:4*b] - q[2*b:3*b]).abs().max().detach().cpu()),
        )
        controller.max_k_copy_error = max(
            controller.max_k_copy_error,
            float((k[b:2*b] - k[:b]).abs().max().detach().cpu()),
            float((k[3*b:4*b] - k[2*b:3*b]).abs().max().detach().cpu()),
        )

    if attention.caching:
        if attention.cached_k is None:
            attention.cached_k, attention.cached_v = k, v
        else:
            attention.cached_k = torch.cat((attention.cached_k, k), dim=2)
            attention.cached_v = torch.cat((attention.cached_v, v), dim=2)
        k, v = attention.cached_k, attention.cached_v

    output = slow_attn(
        query=q,
        key=k,
        value=v,
        scale=attention.scale,
        attn_mask=attn_bias,
        dropout_p=attention.attn_drop if attention.training else 0.0,
    ).transpose(1, 2).reshape(batch4, length, channels)
    return attention.proj_drop(attention.proj(output))


class PaperSACPatch:
    def __init__(self, model, controller):
        self.model = model
        self.controller = controller
        self.original_forwards = []

    def __enter__(self):
        for block in self.model.blocks:
            attention = block.attn
            self.original_forwards.append((attention, attention.forward))
            attention._paper_sac_controller = self.controller
            attention.forward = types.MethodType(_sac_attention_forward, attention)
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        for attention, original_forward in self.original_forwards:
            attention.forward = original_forward
            if hasattr(attention, '_paper_sac_controller'):
                delattr(attention, '_paper_sac_controller')
        return False

## 5. Joint dual-path autoregressive inference

This function implements Algorithm 1 on VAR-CLIP. Both paths share prompt, seed, and random draws. They are exactly identical through $F_3$; only PFB creates divergence. SAC then restores content-path attention geometry during later refinement.

In [ ]:
def prepare_prompt_embedding(prompt):
    tokens = tokenize([prompt]).to(device)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        return clip_model.encode_text(tokens)


def _sample_stream(logits, rng, top_k, top_p):
    return sample_with_top_k_top_p_(
        logits.clone(), rng=rng, top_k=top_k, top_p=top_p, num_samples=1
    )[:, :, 0]


@torch.no_grad()
def paper_dual_path_generate(
    model,
    prompt_embedding,
    style_features,
    *,
    seed=0,
    cfg=4.0,
    top_k=900,
    top_p=0.95,
    pfb_feature_index=2,
    sac_prediction_start=3,
    edit_mode='pfb',
    alpha=1.0,
    rank=None,
    enable_sac=True,
):
    '''Paper Algorithm 1 adapted to VAR-CLIP's 10-stage CFG inference.'''
    base_batch = prompt_embedding.shape[0]
    if base_batch != 1:
        raise ValueError('Demo implementation currently supports one prompt per call.')
    if not 0 <= pfb_feature_index < len(model.patch_nums):
        raise ValueError('Invalid PFB feature index.')
    if enable_sac and not 0 <= sac_prediction_start < len(model.patch_nums):
        raise ValueError('Invalid SAC prediction start.')

    model.eval()
    content_rng = torch.Generator(device=device).manual_seed(seed)
    generation_rng = torch.Generator(device=device).manual_seed(seed)

    null_embedding = model.noise(torch.tensor(0, device=device)).unsqueeze(0).expand(base_batch, -1)
    joint_condition = model.cond_proj(torch.cat((
        prompt_embedding,
        prompt_embedding,
        null_embedding,
        null_embedding,
    ), dim=0))

    level_position = model.lvl_embed(model.lvl_1L) + model.pos_1LC
    next_tokens = (
        joint_condition.unsqueeze(1).expand(4 * base_batch, model.first_l, -1)
        + model.pos_start.expand(4 * base_batch, model.first_l, -1)
        + level_position[:, :model.first_l]
    )

    content_fhat = joint_condition.new_zeros(
        base_batch, model.Cvae, model.patch_nums[-1], model.patch_nums[-1]
    )
    generation_fhat = torch.zeros_like(content_fhat)
    content_trace, generation_trace = [], []

    controller = SACController(base_batch)
    controller.reset_statistics()
    for block in model.blocks:
        block.attn.kv_caching(True)

    current_length = 0
    pre_pfb_max_difference = 0.0
    try:
        with PaperSACPatch(model, controller):
            for step_id, patch_num in enumerate(model.patch_nums):
                controller.active = enable_sac and step_id >= sac_prediction_start
                current_length += patch_num * patch_num

                condition_for_blocks = model.shared_ada_lin(joint_condition)
                hidden = next_tokens
                for block in model.blocks:
                    hidden = block(x=hidden, cond_BD=condition_for_blocks, attn_bias=None)
                all_logits = model.get_logits(hidden, joint_condition)

                ratio = step_id / model.num_stages_minus_1
                cfg_ratio = cfg * ratio
                content_logits = (1 + cfg_ratio) * all_logits[:base_batch] - cfg_ratio * all_logits[2*base_batch:3*base_batch]
                generation_logits = (1 + cfg_ratio) * all_logits[base_batch:2*base_batch] - cfg_ratio * all_logits[3*base_batch:4*base_batch]

                content_indices = _sample_stream(content_logits, content_rng, top_k, top_p)
                generation_indices = _sample_stream(generation_logits, generation_rng, top_k, top_p)

                content_residual = model.vae_quant_proxy[0].embedding(content_indices).transpose(1, 2).reshape(
                    base_batch, model.Cvae, patch_num, patch_num
                )
                generation_residual = model.vae_quant_proxy[0].embedding(generation_indices).transpose(1, 2).reshape(
                    base_batch, model.Cvae, patch_num, patch_num
                )

                content_fhat, content_next = model.vae_quant_proxy[0].get_next_autoregressive_input(
                    step_id, len(model.patch_nums), content_fhat, content_residual
                )
                generation_fhat, generation_next = model.vae_quant_proxy[0].get_next_autoregressive_input(
                    step_id, len(model.patch_nums), generation_fhat, generation_residual
                )

                if step_id <= pfb_feature_index:
                    pre_pfb_max_difference = max(
                        pre_pfb_max_difference,
                        float((content_fhat - generation_fhat).abs().max().detach().cpu()),
                    )

                if step_id == pfb_feature_index and edit_mode != 'none':
                    generation_fhat = apply_feature_edit(
                        generation_fhat,
                        style_features[step_id],
                        mode=edit_mode,
                        alpha=alpha,
                        rank=rank,
                    )
                    if step_id != model.num_stages_minus_1:
                        next_patch = model.patch_nums[step_id + 1]
                        generation_next = F.interpolate(
                            generation_fhat, size=(next_patch, next_patch), mode='area'
                        )

                content_trace.append(content_fhat.detach().clone())
                generation_trace.append(generation_fhat.detach().clone())

                if step_id != model.num_stages_minus_1:
                    next_patch = model.patch_nums[step_id + 1]

                    content_tokens = model.word_embed(
                        content_next.view(base_batch, model.Cvae, -1).transpose(1, 2)
                    ) + level_position[:, current_length:current_length + next_patch ** 2]
                    generation_tokens = model.word_embed(
                        generation_next.view(base_batch, model.Cvae, -1).transpose(1, 2)
                    ) + level_position[:, current_length:current_length + next_patch ** 2]

                    next_tokens = torch.cat((
                        content_tokens,
                        generation_tokens,
                        content_tokens,
                        generation_tokens,
                    ), dim=0)

        content_image = model.vae_proxy[0].fhat_to_img(content_fhat).add(1).mul(0.5)
        generation_image = model.vae_proxy[0].fhat_to_img(generation_fhat).add(1).mul(0.5)
        return {
            'content_image_01': content_image,
            'stylized_image_01': generation_image,
            'content_features': content_trace,
            'generation_features': generation_trace,
            'pre_pfb_max_difference': pre_pfb_max_difference,
            'sac_calls': controller.total_calls,
            'max_q_copy_error': controller.max_q_copy_error,
            'max_k_copy_error': controller.max_k_copy_error,
        }
    finally:
        controller.active = False
        for block in model.blocks:
            block.attn.kv_caching(False)

## 6. Correctness checks

These checks verify implementation behavior before interpreting images:

- both streams are numerically identical before PFB;
- the baseline generation path equals its content path;
- SAC actually runs in all 16 transformer blocks for every selected later scale;
- copied Q/K tensors have zero numerical mismatch.

In [ ]:
PAPER_ALPHA = 1.0
PAPER_PFB_FEATURE_INDEX = 2      # F_3: cumulative feature after 3x3 residual
PAPER_SAC_PREDICTION_START = 3   # first causal consumer of edited F_3 predicts R_4
CFG = 4.0
TOP_K = 900
TOP_P = 0.95
SEED = 0

smoke_case = DEMO_CASES[0]
smoke_m11, smoke_style_01, _ = load_reference_image(smoke_case['style_path'])
smoke_style_features = extract_multiscale_style_features(smoke_m11)
smoke_prompt_embedding = prepare_prompt_embedding(smoke_case['prompt'])

with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
    smoke_baseline = paper_dual_path_generate(
        var_clip,
        smoke_prompt_embedding,
        smoke_style_features,
        seed=SEED,
        cfg=CFG,
        top_k=TOP_K,
        top_p=TOP_P,
        edit_mode='none',
        enable_sac=False,
    )
    smoke_method = paper_dual_path_generate(
        var_clip,
        smoke_prompt_embedding,
        smoke_style_features,
        seed=SEED,
        cfg=CFG,
        top_k=TOP_K,
        top_p=TOP_P,
        pfb_feature_index=PAPER_PFB_FEATURE_INDEX,
        sac_prediction_start=PAPER_SAC_PREDICTION_START,
        edit_mode='pfb',
        alpha=PAPER_ALPHA,
        rank=None,
        enable_sac=True,
    )

expected_sac_calls = len(var_clip.blocks) * (len(PATCH_NUMS) - PAPER_SAC_PREDICTION_START)
baseline_stream_error = float(
    (smoke_baseline['content_image_01'] - smoke_baseline['stylized_image_01']).abs().max().cpu()
)
checks = {
    'baseline_content_generation_max_error': baseline_stream_error,
    'pre_pfb_stream_max_error': smoke_method['pre_pfb_max_difference'],
    'SAC_calls': smoke_method['sac_calls'],
    'expected_SAC_calls': expected_sac_calls,
    'max_Q_copy_error': smoke_method['max_q_copy_error'],
    'max_K_copy_error': smoke_method['max_k_copy_error'],
}
print(checks)

assert baseline_stream_error == 0.0
assert smoke_method['pre_pfb_max_difference'] == 0.0
assert smoke_method['sac_calls'] == expected_sac_calls
assert smoke_method['max_q_copy_error'] == 0.0
assert smoke_method['max_k_copy_error'] == 0.0
print('All paper-port invariants passed.')

## 7. Demo suite

Each row shows the style reference, unmodified VAR-CLIP content path, PFB only, and the full PFB + SAC method. All methods use the same prompt and seed. The content path is the correct baseline; it is not a VAE reconstruction or token-mixed image.

In [ ]:
RUN_DEMO_SUITE = True
DEMO_CASE_LIMIT = 5
RUN_PFB_ONLY = True


def show_demo_rows(rows, save_path=None):
    columns = ['style reference', 'VAR-CLIP baseline', 'PFB only', 'PFB + SAC']
    figure, axes = plt.subplots(len(rows), len(columns), figsize=(4 * len(columns), 4 * len(rows)))
    if len(rows) == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_id, row in enumerate(rows):
        images = [row['style'], row['baseline'], row['pfb_only'], row['pfb_sac']]
        for col_id, (column, image) in enumerate(zip(columns, images)):
            axis = axes[row_id, col_id]
            tensor = image[0].detach().float().cpu().clamp(0, 1)
            axis.imshow(tensor.permute(1, 2, 0).numpy())
            if row_id == 0:
                axis.set_title(column, fontsize=12)
            if col_id == 0:
                axis.set_ylabel(row['name'] + '\n' + row['prompt'], fontsize=10)
            axis.set_xticks([])
            axis.set_yticks([])

    figure.suptitle('Paper-faithful SVD PFB + SAC port to VAR-CLIP', fontsize=15)
    figure.tight_layout()
    if save_path is not None:
        figure.savefig(save_path, dpi=180, bbox_inches='tight')
        print('saved:', save_path)
    plt.show()


demo_rows = []
if RUN_DEMO_SUITE:
    for case_id, current_case in enumerate(tqdm(DEMO_CASES[:DEMO_CASE_LIMIT], desc='Paper-method demos')):
        style_m11, style_01, _ = load_reference_image(current_case['style_path'])
        prompt_embedding = prepare_prompt_embedding(current_case['prompt'])
        with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
            style_features = extract_multiscale_style_features(style_m11)
            full_result = paper_dual_path_generate(
                var_clip,
                prompt_embedding,
                style_features,
                seed=SEED,
                cfg=CFG,
                top_k=TOP_K,
                top_p=TOP_P,
                pfb_feature_index=PAPER_PFB_FEATURE_INDEX,
                sac_prediction_start=PAPER_SAC_PREDICTION_START,
                edit_mode='pfb',
                alpha=PAPER_ALPHA,
                rank=None,
                enable_sac=True,
            )
            if RUN_PFB_ONLY:
                pfb_result = paper_dual_path_generate(
                    var_clip,
                    prompt_embedding,
                    style_features,
                    seed=SEED,
                    cfg=CFG,
                    top_k=TOP_K,
                    top_p=TOP_P,
                    pfb_feature_index=PAPER_PFB_FEATURE_INDEX,
                    sac_prediction_start=PAPER_SAC_PREDICTION_START,
                    edit_mode='pfb',
                    alpha=PAPER_ALPHA,
                    rank=None,
                    enable_sac=False,
                )
            else:
                pfb_result = full_result

        demo_rows.append({
            'name': current_case['name'],
            'prompt': current_case['prompt'],
            'style': style_01,
            'baseline': full_result['content_image_01'],
            'pfb_only': pfb_result['stylized_image_01'],
            'pfb_sac': full_result['stylized_image_01'],
        })
        del style_features, full_result, pfb_result
        gc.collect()
        torch.cuda.empty_cache()

    show_demo_rows(demo_rows, OUTPUT_DIR / f'demo_suite_seed{SEED}.png')

## 8. CLIP metrics

The paper reports text-prompt similarity $S_{txt}$, reference-image similarity $S_{img}$, and their harmonic mean. These scores are useful for comparing variants, but $S_{img}$ alone can reward content leakage. Always inspect the images beside the table.

In [ ]:
CLIP_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=device).view(1, 3, 1, 1)
CLIP_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=device).view(1, 3, 1, 1)


def clip_image_embedding(image_01):
    resized = F.interpolate(image_01.to(device), size=(224, 224), mode='bicubic', align_corners=False)
    normalized = (resized.clamp(0, 1) - CLIP_MEAN) / CLIP_STD
    return clip_model.encode_image(normalized)


def cosine_score(a, b):
    return float((a * b).sum(dim=-1).mean().detach().cpu())


metric_rows = []
if RUN_DEMO_SUITE:
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        for row in demo_rows:
            prompt_embedding = prepare_prompt_embedding(row['prompt'])
            style_embedding = clip_image_embedding(row['style'])
            for method in ('baseline', 'pfb_only', 'pfb_sac'):
                output_embedding = clip_image_embedding(row[method])
                s_txt = cosine_score(output_embedding, prompt_embedding)
                s_img = cosine_score(output_embedding, style_embedding)
                harmonic = 2 * s_txt * s_img / max(s_txt + s_img, 1e-8)
                metric_rows.append({
                    'case': row['name'],
                    'method': method,
                    'S_txt': s_txt,
                    'S_img': s_img,
                    'S_harmonic': harmonic,
                })

    metric_table = pd.DataFrame(metric_rows)
    display(metric_table.round(4))
    display(metric_table.groupby('method')[['S_txt', 'S_img', 'S_harmonic']].mean().round(4))
    metric_table.to_csv(OUTPUT_DIR / f'demo_metrics_seed{SEED}.csv', index=False)

## 9. Optional paper ablations

Enable these only after the five-case demo succeeds. Defaults match the supplementary material:

- exponential decay $\alpha \in \{0.2,0.6,1.0,2.0,5.0\}$;
- PFB intervention at every VAR-CLIP scale to verify whether its pivotal feature is also $F_3$;
- component comparison: baseline, full replacement, PFB, and PFB + SAC.

The paper discovered $F_3$ on Infinity through an empirical step sweep. It is a hypothesis, not a guarantee, that VAR-CLIP has the same pivotal scale. The step ablation is therefore scientifically necessary for this backbone port.

In [ ]:
RUN_ALPHA_ABLATION = False
RUN_STEP_ABLATION = False
RUN_COMPONENT_ABLATION = False

ABLATION_CASE = DEMO_CASES[0]
ABLATION_SEED = 0
ALPHAS = [0.2, 0.6, 1.0, 2.0, 5.0]


def run_variant(style_features, prompt_embedding, **overrides):
    arguments = dict(
        seed=ABLATION_SEED,
        cfg=CFG,
        top_k=TOP_K,
        top_p=TOP_P,
        pfb_feature_index=PAPER_PFB_FEATURE_INDEX,
        sac_prediction_start=PAPER_SAC_PREDICTION_START,
        edit_mode='pfb',
        alpha=PAPER_ALPHA,
        rank=None,
        enable_sac=True,
    )
    arguments.update(overrides)
    with torch.inference_mode(), torch.autocast('cuda', dtype=torch.float16):
        return paper_dual_path_generate(var_clip, prompt_embedding, style_features, **arguments)


if RUN_ALPHA_ABLATION or RUN_STEP_ABLATION or RUN_COMPONENT_ABLATION:
    ablation_m11, ablation_style_01, _ = load_reference_image(ABLATION_CASE['style_path'])
    ablation_style_features = extract_multiscale_style_features(ablation_m11)
    ablation_prompt_embedding = prepare_prompt_embedding(ABLATION_CASE['prompt'])

if RUN_ALPHA_ABLATION:
    alpha_items = []
    for alpha in tqdm(ALPHAS, desc='Alpha ablation'):
        result = run_variant(ablation_style_features, ablation_prompt_embedding, alpha=alpha)
        alpha_items.append((f'alpha={alpha}', result['stylized_image_01']))

    figure, axes = plt.subplots(1, len(alpha_items), figsize=(4 * len(alpha_items), 4))
    for axis, (title, image) in zip(axes, alpha_items):
        axis.imshow(image[0].float().cpu().clamp(0, 1).permute(1, 2, 0))
        axis.set_title(title)
        axis.axis('off')
    plt.tight_layout()
    plt.show()

if RUN_STEP_ABLATION:
    step_items = []
    for step_id, patch_num in enumerate(tqdm(PATCH_NUMS, desc='PFB step ablation')):
        sac_start = min(step_id + 1, len(PATCH_NUMS) - 1)
        result = run_variant(
            ablation_style_features,
            ablation_prompt_embedding,
            pfb_feature_index=step_id,
            sac_prediction_start=sac_start,
            enable_sac=step_id < len(PATCH_NUMS) - 1,
        )
        step_items.append((f'F_{step_id + 1} ({patch_num}x{patch_num})', result['stylized_image_01']))

    figure, axes = plt.subplots(2, 5, figsize=(20, 8))
    for axis, (title, image) in zip(axes.flat, step_items):
        axis.imshow(image[0].float().cpu().clamp(0, 1).permute(1, 2, 0))
        axis.set_title(title)
        axis.axis('off')
    plt.tight_layout()
    plt.show()

if RUN_COMPONENT_ABLATION:
    baseline = run_variant(
        ablation_style_features, ablation_prompt_embedding, edit_mode='none', enable_sac=False
    )
    replacement = run_variant(
        ablation_style_features, ablation_prompt_embedding, edit_mode='replace', enable_sac=False
    )
    pfb_only = run_variant(
        ablation_style_features, ablation_prompt_embedding, edit_mode='pfb', enable_sac=False
    )
    full_method = run_variant(
        ablation_style_features, ablation_prompt_embedding, edit_mode='pfb', enable_sac=True
    )

    component_items = [
        ('style reference', ablation_style_01),
        ('baseline', baseline['content_image_01']),
        ('full replacement', replacement['stylized_image_01']),
        ('PFB only', pfb_only['stylized_image_01']),
        ('PFB + SAC', full_method['stylized_image_01']),
    ]
    figure, axes = plt.subplots(1, len(component_items), figsize=(4 * len(component_items), 4))
    for axis, (title, image) in zip(axes, component_items):
        axis.imshow(image[0].float().cpu().clamp(0, 1).permute(1, 2, 0))
        axis.set_title(title)
        axis.axis('off')
    plt.tight_layout()
    plt.show()

## What this notebook can establish

A successful run establishes whether the paper's inference intervention transfers to VAR-CLIP as an architecture-level idea. It does not establish parity with Infinity-2B. Interpret results in this order:

1. Verify baseline prompt quality. A weak VAR-CLIP baseline limits every edited result.
2. Check whether PFB increases visible style while preserving the prompted object.
3. Check whether SAC repairs geometry lost by PFB without erasing style.
4. Use the step sweep to locate VAR-CLIP's own pivotal feature instead of assuming Infinity's $F_3$ is universal.
5. Treat higher reference-image CLIP similarity cautiously because it can reward leaked reference content.